In [67]:
import numpy as np
import pandas as pd
import torch
import folium
from IPython.display import display

from lstm import lstm_seq2seq

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [68]:
feature_cols = [
    "LAT",
    "LON",
    "SPEED",
    "dt",
    "cog_sin",
    "cog_cos",
    "hdg_sin",
    "hdg_cos",
    "dlat",
    "dlon",
]

target_cols = ["dlat", "dlon"]

SEQ_LEN = 5
TARGET_LEN = 1

HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2

In [69]:
df_test = pd.read_csv("../data/vessel_prediction_model_data/mid_model_data/mid_test_data.csv")
df_train = pd.read_csv("../data/vessel_prediction_model_data/mid_model_data/mid_train_data.csv")

In [70]:
feature_means = df_train[feature_cols].mean()
feature_stds  = df_train[feature_cols].std()

target_means = df_train[target_cols].mean()
target_stds  = df_train[target_cols].std()

In [71]:
df_test[feature_cols] = (df_test[feature_cols] - feature_means) / feature_stds
df_test[target_cols]  = (df_test[target_cols]  - target_means) / target_stds

In [72]:
def make_windows_from_df(df, feature_cols, target_cols, seq_len=5, target_len=1,
                         route_col="voyage_id", time_col="TIME"):
    X_list = []
    Y_list = []
    meta_list = []

    for route_id, group in df.groupby(route_col):
        group = group.sort_values(time_col).reset_index(drop=True)

        feat = group[feature_cols].to_numpy(dtype=np.float32)
        targ = group[target_cols].to_numpy(dtype=np.float32)

        n = len(group)
        window_count = n - seq_len - target_len + 1
        if window_count <= 0:
            continue

        for i in range(window_count):
            x_win = feat[i:i+seq_len]
            y_win = targ[i+seq_len:i+seq_len+target_len]

            X_list.append(x_win)
            Y_list.append(y_win)

            meta_list.append({
                "voyage_id": route_id,
                "start_idx": i,
                "last_known_time": group.iloc[i + seq_len - 1][time_col],
            })

    if len(X_list) == 0:
        raise ValueError("No windows were created.")

    X = np.stack(X_list, axis=0)   # (N, seq_len, features)
    Y = np.stack(Y_list, axis=0)   # (N, target_len, 2)

    X = np.transpose(X, (1, 0, 2)) # (seq_len, N, features)
    Y = np.transpose(Y, (1, 0, 2)) # (target_len, N, 2)

    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)

    return X, Y, meta_list

In [73]:
X_test, Y_test, meta_test = make_windows_from_df(
    df_test,
    feature_cols,
    target_cols,
    seq_len=SEQ_LEN,
    target_len=TARGET_LEN,
    route_col="voyage_id",
    time_col="TIME"
)

In [74]:
X_test = torch.tensor(X_test, dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.float32)

C:\Users\logan\AppData\Local\Temp\ipykernel_23376\2696808717.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
C:\Users\logan\AppData\Local\Temp\ipykernel_23376\2696808717.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y_test = torch.tensor(Y_test, dtype=torch.float32)


In [75]:
model = lstm_seq2seq(
    input_size=len(feature_cols),
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    target_indices=(0, 1),
    decoder_feature_indices=tuple(range(len(feature_cols))),
    predict_deltas=True,
).to(DEVICE)

model.load_state_dict(torch.load("best_lstm_seq2seq_dlat_dlon.pt", map_location=DEVICE))
model.eval()

C:\Users\logan\AppData\Local\Temp\ipykernel_23376\1553603474.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_lstm_seq2seq_dlat_dl

lstm_seq2seq(
  (encoder): lstm_encoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
  )
  (decoder): lstm_decoder(
    (lstm): LSTM(10, 128, num_layers=2, dropout=0.2)
    (linear): Linear(in_features=128, out_features=2, bias=True)
  )
  (target_to_decoder): Linear(in_features=2, out_features=10, bias=True)
)

In [76]:
with torch.no_grad():
    Y_pred = model.predict(
        X_test.to(DEVICE),
        target_len=TARGET_LEN,
        return_absolute=False
    )

In [77]:
def unscale_deltas(delta_tensor, target_means, target_stds):
    if not torch.is_tensor(delta_tensor):
        delta_tensor = torch.tensor(delta_tensor, dtype=torch.float32)
    else:
        delta_tensor = delta_tensor.to(dtype=torch.float32)

    means = torch.tensor(
        [float(target_means["dlat"]), float(target_means["dlon"])],
        dtype=torch.float32,
        device=delta_tensor.device
    ).view(1, 1, 2)

    stds = torch.tensor(
        [float(target_stds["dlat"]), float(target_stds["dlon"])],
        dtype=torch.float32,
        device=delta_tensor.device
    ).view(1, 1, 2)

    return delta_tensor * stds + means


def get_last_known_abs_point(X_test, sample_idx, feature_means, feature_stds, feature_cols):
    lat_idx = feature_cols.index("LAT")
    lon_idx = feature_cols.index("LON")

    last_lat_scaled = X_test[-1, sample_idx, lat_idx].item()
    last_lon_scaled = X_test[-1, sample_idx, lon_idx].item()

    last_lat = last_lat_scaled * float(feature_stds["LAT"]) + float(feature_means["LAT"])
    last_lon = last_lon_scaled * float(feature_stds["LON"]) + float(feature_means["LON"])

    return last_lat, last_lon


def delta_to_absolute(last_lat, last_lon, dlat, dlon):
    return last_lat + dlat, last_lon + dlon

In [78]:
def metrics_in_yards(pred_scaled, true_scaled, X_tensor,
                     feature_means, feature_stds,
                     target_means, target_stds,
                     feature_cols):

    # force everything to torch
    if not torch.is_tensor(pred_scaled):
        pred_scaled = torch.tensor(pred_scaled, dtype=torch.float32)
    else:
        pred_scaled = pred_scaled.to(dtype=torch.float32)

    if not torch.is_tensor(true_scaled):
        true_scaled = torch.tensor(true_scaled, dtype=torch.float32)
    else:
        true_scaled = true_scaled.to(dtype=torch.float32)

    if not torch.is_tensor(X_tensor):
        X_tensor = torch.tensor(X_tensor, dtype=torch.float32)
    else:
        X_tensor = X_tensor.to(dtype=torch.float32)

    device = pred_scaled.device

    true_scaled = true_scaled.to(device)
    X_tensor = X_tensor.to(device)

    pred_deg = unscale_deltas(pred_scaled, target_means, target_stds, device=device)
    true_deg = unscale_deltas(true_scaled, target_means, target_stds, device=device)

    err_deg = pred_deg - true_deg
    err_dlat_deg = err_deg[:, :, 0]
    err_dlon_deg = err_deg[:, :, 1]

    lat_idx = feature_cols.index("LAT")
    last_lat_scaled = X_tensor[-1, :, lat_idx]
    last_lat_deg = last_lat_scaled * float(feature_stds["LAT"]) + float(feature_means["LAT"])
    last_lat_deg = last_lat_deg.unsqueeze(0).expand(err_dlat_deg.shape[0], -1)

    meters_per_deg_lat = 111320.0
    meters_per_deg_lon = 111320.0 * torch.cos(torch.deg2rad(last_lat_deg))

    err_north_m = err_dlat_deg * meters_per_deg_lat
    err_east_m  = err_dlon_deg * meters_per_deg_lon

    err_m = torch.sqrt(err_north_m**2 + err_east_m**2)
    err_yards = err_m * 1.0936133

    mse_yards2 = torch.mean(err_yards ** 2).item()
    rmse_yards = torch.sqrt(torch.mean(err_yards ** 2)).item()
    mae_yards = torch.mean(err_yards).item()

    return mse_yards2, rmse_yards, mae_yards

In [79]:
mse_yards2, rmse_yards, mae_yards = metrics_in_yards(
    Y_pred,
    Y_test.to(DEVICE),
    X_test.to(DEVICE),
    feature_means,
    feature_stds,
    target_means,
    target_stds,
    feature_cols
)

print("MSE (yards^2):", mse_yards2)
print("RMSE (yards):", rmse_yards)
print("MAE (yards):", mae_yards)

TypeError: unscale_deltas() got an unexpected keyword argument 'device'

In [80]:
def map_single_prediction(sample_idx, X_test, Y_test, Y_pred,
                          meta_test, feature_means, feature_stds,
                          target_means, target_stds, feature_cols):
    # make sure tensors
    if not torch.is_tensor(X_test):
        X_test_local = torch.tensor(X_test, dtype=torch.float32)
    else:
        X_test_local = X_test.detach().cpu()

    if not torch.is_tensor(Y_test):
        Y_test_local = torch.tensor(Y_test, dtype=torch.float32)
    else:
        Y_test_local = Y_test.detach().cpu()

    if not torch.is_tensor(Y_pred):
        Y_pred_local = torch.tensor(Y_pred, dtype=torch.float32)
    else:
        Y_pred_local = Y_pred.detach().cpu()

    # unscale deltas
    Y_pred_unscaled = unscale_deltas(Y_pred_local, target_means, target_stds).cpu()
    Y_test_unscaled = unscale_deltas(Y_test_local, target_means, target_stds).cpu()

    # last known absolute point
    last_lat, last_lon = get_last_known_abs_point(
        X_test_local, sample_idx, feature_means, feature_stds, feature_cols
    )

    # target_len=1, so use [0, sample_idx]
    pred_dlat = Y_pred_unscaled[0, sample_idx, 0].item()
    pred_dlon = Y_pred_unscaled[0, sample_idx, 1].item()

    true_dlat = Y_test_unscaled[0, sample_idx, 0].item()
    true_dlon = Y_test_unscaled[0, sample_idx, 1].item()

    pred_lat, pred_lon = delta_to_absolute(last_lat, last_lon, pred_dlat, pred_dlon)
    true_lat, true_lon = delta_to_absolute(last_lat, last_lon, true_dlat, true_dlon)

    voyage_id = meta_test[sample_idx]["voyage_id"] if meta_test is not None else "unknown"
    last_time = meta_test[sample_idx].get("last_known_time", "unknown") if meta_test is not None else "unknown"

    center_lat = np.mean([last_lat, pred_lat, true_lat])
    center_lon = np.mean([last_lon, pred_lon, true_lon])

    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

    # Last known AIS point
    folium.CircleMarker(
        location=[last_lat, last_lon],
        radius=7,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.9,
        popup=f"Voyage: {voyage_id}<br>Last known AIS<br>Time: {last_time}<br>Lat: {last_lat:.6f}<br>Lon: {last_lon:.6f}",
        tooltip="Last known AIS"
    ).add_to(m)

    # Predicted point
    folium.CircleMarker(
        location=[pred_lat, pred_lon],
        radius=7,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.9,
        popup=f"Voyage: {voyage_id}<br>Predicted point<br>Lat: {pred_lat:.6f}<br>Lon: {pred_lon:.6f}",
        tooltip="Predicted point"
    ).add_to(m)

    # Actual point
    folium.CircleMarker(
        location=[true_lat, true_lon],
        radius=7,
        color="green",
        fill=True,
        fill_color="green",
        fill_opacity=0.9,
        popup=f"Voyage: {voyage_id}<br>Actual point<br>Lat: {true_lat:.6f}<br>Lon: {true_lon:.6f}",
        tooltip="Actual point"
    ).add_to(m)

    # lines from last known point
    folium.PolyLine(
        [(last_lat, last_lon), (pred_lat, pred_lon)],
        weight=2,
        color="red",
        opacity=0.8,
        tooltip="Predicted path"
    ).add_to(m)

    folium.PolyLine(
        [(last_lat, last_lon), (true_lat, true_lon)],
        weight=2,
        color="green",
        opacity=0.8,
        tooltip="Actual path"
    ).add_to(m)

    return m

In [81]:
from ipywidgets import interact, IntSlider
from IPython.display import display

def show_prediction(sample_idx):
    print(f"Sample index: {sample_idx}")
    if meta_test is not None:
        print("Voyage:", meta_test[sample_idx]["voyage_id"])
        print("Last known time:", meta_test[sample_idx].get("last_known_time", "unknown"))

    m = map_single_prediction(
        sample_idx,
        X_test,
        Y_test,
        Y_pred,
        meta_test,
        feature_means,
        feature_stds,
        target_means,
        target_stds,
        feature_cols
    )
    display(m)

interact(
    show_prediction,
    sample_idx=IntSlider(min=0, max=X_test.shape[1] - 1, step=1, value=0)
)

ModuleNotFoundError: No module named 'ipywidgets'

In [82]:
m = map_single_prediction(
    0,
    X_test,
    Y_test,
    Y_pred,
    meta_test,
    feature_means,
    feature_stds,
    target_means,
    target_stds,
    feature_cols
)

m